In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as sp_stats
from scipy.stats import binomtest, mannwhitneyu, fisher_exact, kruskal
from IPython.display import display, HTML, Markdown

# ── Database connection ──
DB_PATH = "C:/Users/scgee/OneDrive/Documents/Projects/PatientPunk/polina_onemonth.db"
conn = sqlite3.connect(DB_PATH)

# ── Sentiment mapping ──
SENTIMENT_SCORE = {"positive": 1.0, "mixed": 0.5, "neutral": 0.0, "negative": -1.0}

def to_numeric(s):
    """Convert sentiment string to numeric score."""
    return SENTIMENT_SCORE.get(s, 0.0)

def classify_outcome(avg_score):
    """Classify user-level average into outcome category."""
    if avg_score > 0.7:
        return "positive"
    elif avg_score < -0.3:
        return "negative"
    return "mixed/neutral"

def wilson_ci(k, n, z=1.96):
    """Wilson score confidence interval for a proportion."""
    if n == 0:
        return 0.0, 0.0
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    margin = z * np.sqrt((p * (1 - p) + z**2 / (4 * n)) / n) / denom
    return max(0, center - margin), min(1, center + margin)

def nnt(treatment_rate, baseline_rate):
    """Number needed to treat. Returns None if rates are equal or inverted."""
    diff = treatment_rate - baseline_rate
    if diff <= 0:
        return None
    return round(1 / diff, 1)

# ── Chart defaults ──
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

# ── Filtering sets ──
GENERIC_TERMS = {
    "supplements", "medication", "treatment", "therapy", "drug", "drugs",
    "vitamin", "prescription", "pill", "pills", "dosage", "dose",
}

# Colors
COLORS = {"positive": "#2ecc71", "mixed/neutral": "#95a5a6", "negative": "#e74c3c"}


**Research Question:** "How do POTS patients in the Long COVID community compare to the broader population in terms of comorbidity burden, treatment patterns, symptom experience, and treatment outcomes?"

## Abstract

Postural Orthostatic Tachycardia Syndrome (POTS) -- a form of dysautonomia characterized by excessive heart rate increase upon standing -- is one of the most discussed comorbidities in the Long COVID community. This analysis compares 80 POTS-identified users against 2,747 non-POTS users from r/covidlonghaulers over a one-month period (March-April 2026). POTS users carry a significantly heavier comorbidity burden (median 8 vs 2 conditions), try more treatments (median 7 vs 3), post 3.6x more frequently, and cluster tightly with MCAS and dysautonomia in a recognizable triad pattern. Despite trying more treatments, POTS users report a lower overall positive sentiment rate (56% vs 68%, Fisher exact p<0.05), suggesting harder-to-treat symptomatology. This is a preliminary survey study with modest sample sizes; findings should be treated as hypotheses for follow-up investigation.

## 1. Data Exploration

Data covers: **2026-03-11 to 2026-04-10** (1 month), sourced from r/covidlonghaulers.

| Metric | Count |
|--------|-------|
| Total users | 2,827 |
| Total posts | 17,182 |
| Treatment reports | 6,815 |
| Unique reporters | 1,121 |
| POTS-identified users | 80 (2.8%) |
| POTS users with treatment reports | 51 (63.8%) |

POTS identification is based on condition extraction from user posts -- users whose posts mention POTS or postural orthostatic tachycardia in a diagnostic context. This is a text-mining label, not a verified clinical diagnosis.

## 2. The POTS Cohort: Engagement and Burden

Before comparing treatment outcomes, we need to understand who these POTS users are. Are they typical community members who happen to have POTS, or are they a systematically different subgroup?

In [ ]:

# Identify POTS user IDs
pots_ids = set(pd.read_sql('''
    SELECT DISTINCT user_id FROM conditions
    WHERE LOWER(condition_name) LIKE '%pots%'
    OR LOWER(condition_name) LIKE '%postural%tachycardia%'
''', conn)['user_id'])

# Condition count per user
cond_per_user = pd.read_sql('''
    SELECT user_id, COUNT(DISTINCT condition_name) as n_conditions
    FROM conditions GROUP BY user_id
''', conn)
cond_per_user['group'] = cond_per_user['user_id'].apply(lambda x: 'POTS' if x in pots_ids else 'non-POTS')

# Treatment count per user (among reporters)
tx_per_user = pd.read_sql('''
    SELECT user_id, COUNT(DISTINCT drug_id) as n_treatments
    FROM treatment_reports GROUP BY user_id
''', conn)
tx_per_user['group'] = tx_per_user['user_id'].apply(lambda x: 'POTS' if x in pots_ids else 'non-POTS')

# Post count per user
posts_per_user = pd.read_sql('''
    SELECT user_id, COUNT(*) as n_posts
    FROM posts GROUP BY user_id
''', conn)
posts_per_user['group'] = posts_per_user['user_id'].apply(lambda x: 'POTS' if x in pots_ids else 'non-POTS')

# Build comparison table
rows_data = []
for metric, df, col in [('Conditions per user', cond_per_user, 'n_conditions'),
                          ('Treatments tried', tx_per_user, 'n_treatments'),
                          ('Posts per user', posts_per_user, 'n_posts')]:
    for grp in ['POTS', 'non-POTS']:
        subset = df[df['group'] == grp][col]
        rows_data.append({
            'Metric': metric, 'Group': grp, 'n': len(subset),
            'Median': subset.median(), 'Mean': round(subset.mean(), 1),
            'IQR': f"{subset.quantile(0.25):.0f} - {subset.quantile(0.75):.0f}"
        })

comp_df = pd.DataFrame(rows_data)

# Mann-Whitney U tests
test_results = []
for metric, df, col in [('Conditions per user', cond_per_user, 'n_conditions'),
                          ('Treatments tried', tx_per_user, 'n_treatments'),
                          ('Posts per user', posts_per_user, 'n_posts')]:
    pots_vals = df[df['group'] == 'POTS'][col].values
    nonpots_vals = df[df['group'] == 'non-POTS'][col].values
    stat, p = mannwhitneyu(pots_vals, nonpots_vals, alternative='two-sided')
    n1, n2 = len(pots_vals), len(nonpots_vals)
    r = 1 - (2 * stat) / (n1 * n2)
    test_results.append({'Metric': metric, 'U': f"{stat:.0f}",
                          'p-value': f"{p:.2e}" if p < 0.001 else f"{p:.4f}",
                          'rank-biserial r': f"{r:.3f}"})

display(comp_df.style.set_caption('POTS vs non-POTS: Engagement and Burden').hide(axis='index'))
display(pd.DataFrame(test_results).style.set_caption('Mann-Whitney U tests (two-sided)').hide(axis='index'))


**What this means:** POTS users are not typical community members. They carry a median of 8 co-occurring conditions compared to 2 for non-POTS users, try roughly twice as many treatments, and post 3-4 times more frequently. All three differences are statistically significant with large effect sizes. This could reflect genuinely more complex illness, greater health literacy and engagement, or both.

In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

for i, (metric_name, df, col, title) in enumerate([
    ('Conditions', cond_per_user, 'n_conditions', 'Co-occurring Conditions'),
    ('Treatments', tx_per_user, 'n_treatments', 'Treatments Tried'),
    ('Posts', posts_per_user, 'n_posts', 'Posts per User')]):
    ax = axes[i]
    pots_v = df[df['group'] == 'POTS'][col]
    nonpots_v = df[df['group'] == 'non-POTS'][col]

    data_plot = pd.DataFrame({
        'Group': ['POTS', 'non-POTS'],
        'Median': [pots_v.median(), nonpots_v.median()],
        'Q25': [pots_v.quantile(0.25), nonpots_v.quantile(0.25)],
        'Q75': [pots_v.quantile(0.75), nonpots_v.quantile(0.75)],
        'n': [len(pots_v), len(nonpots_v)]
    })

    colors_bar = ['#3498db', '#bdc3c7']
    bars = ax.bar(data_plot['Group'], data_plot['Median'], color=colors_bar, width=0.5, edgecolor='white')
    ax.errorbar(data_plot['Group'], data_plot['Median'],
                yerr=[data_plot['Median'] - data_plot['Q25'], data_plot['Q75'] - data_plot['Median']],
                fmt='none', color='black', capsize=5, linewidth=1.5)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel('Median (IQR)')
    for j, bar in enumerate(bars):
        yval = data_plot['Q75'].iloc[j] + 0.5
        ax.text(bar.get_x() + bar.get_width()/2, yval,
                f"n={data_plot['n'].iloc[j]}", ha='center', va='bottom', fontsize=9, color='#555')

fig.suptitle('POTS Users Are Heavier Engagers Across All Metrics', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


**Key takeaway:** The engagement gap is striking. POTS users are among the most active, most burdened, and most treatment-experienced members of the Long COVID community. This is consistent with POTS being a condition that drives sustained healthcare-seeking behavior.

## 3. The Comorbidity Triad: POTS, MCAS, and Dysautonomia

POTS users carry far more co-occurring conditions. But which conditions cluster together? The Long COVID community frequently discusses the "POTS-MCAS-EDS triad" -- does the data support this pattern?

In [ ]:

# Conditions for POTS users (filter community-defining long covid)
pots_conditions = pd.read_sql('''
    SELECT DISTINCT user_id, condition_name
    FROM conditions
    WHERE user_id IN (
        SELECT DISTINCT user_id FROM conditions
        WHERE LOWER(condition_name) LIKE '%pots%'
        OR LOWER(condition_name) LIKE '%postural%tachycardia%'
    )
    AND LOWER(condition_name) NOT LIKE '%pots%'
    AND LOWER(condition_name) NOT LIKE '%postural%tachycardia%'
    AND LOWER(condition_name) NOT IN ('long covid', 'covid related', 'covid induced')
''', conn)

# Co-occurrence rates
cooccur = pots_conditions.groupby('condition_name')['user_id'].nunique().reset_index()
cooccur.columns = ['Condition', 'POTS Users (n=80)']
cooccur['pct_pots'] = (cooccur['POTS Users (n=80)'] / 80 * 100).round(1)

# non-POTS rates
nonpots_conditions = pd.read_sql('''
    SELECT DISTINCT user_id, condition_name
    FROM conditions
    WHERE user_id NOT IN (
        SELECT DISTINCT user_id FROM conditions
        WHERE LOWER(condition_name) LIKE '%pots%'
        OR LOWER(condition_name) LIKE '%postural%tachycardia%'
    )
    AND LOWER(condition_name) NOT IN ('long covid', 'covid related', 'covid induced')
''', conn)

nonpots_cooccur = nonpots_conditions.groupby('condition_name')['user_id'].nunique().reset_index()
nonpots_cooccur.columns = ['Condition', 'non-POTS Users']
total_nonpots_cond = nonpots_conditions['user_id'].nunique()

merged = cooccur.merge(nonpots_cooccur, on='Condition', how='left').fillna(0)
merged['pct_nonpots'] = (merged['non-POTS Users'] / total_nonpots_cond * 100).round(1)
merged['Fold enrichment'] = (merged['pct_pots'] / merged['pct_nonpots'].replace(0, 0.1)).round(1)
merged = merged.sort_values('POTS Users (n=80)', ascending=False).head(12)

# Fisher exact for each
for idx, row in merged.iterrows():
    pots_yes = int(row['POTS Users (n=80)'])
    pots_no = 80 - pots_yes
    np_yes = int(row['non-POTS Users'])
    np_no = total_nonpots_cond - np_yes
    _, p_val = fisher_exact([[pots_yes, pots_no], [np_yes, np_no]])
    merged.loc[idx, 'p-value'] = p_val

merged['p_display'] = merged['p-value'].apply(lambda x: f"{x:.2e}" if x < 0.001 else f"{x:.4f}")

display(merged[['Condition', 'POTS Users (n=80)', 'pct_pots', 'pct_nonpots', 'Fold enrichment', 'p_display']].rename(
    columns={'pct_pots': '% of POTS', 'pct_nonpots': '% of non-POTS', 'p_display': 'p-value'}
).style.set_caption('Conditions Co-occurring with POTS (excluding Long COVID)').hide(axis='index')
.format({'% of POTS': '{:.1f}%', '% of non-POTS': '{:.1f}%', 'Fold enrichment': '{:.1f}x'}))


**What this means:** Three conditions stand out as dramatically enriched in the POTS cohort: MCAS (Mast Cell Activation Syndrome), dysautonomia, and ME/CFS (Myalgic Encephalomyelitis/Chronic Fatigue Syndrome). PEM (Post-Exertional Malaise) is also extremely common at 80%. EDS (Ehlers-Danlos Syndrome) co-occurs with POTS in about a third of cases, consistent with the known clinical association. All enrichments are statistically significant.

In [ ]:

top_conds = ['pem', 'mcas', 'dysautonomia', 'me/cfs', 'post-viral',
             'ehlers-danlos syndrome', 'ibs', 'fibromyalgia',
             'small fiber neuropathy', 'mast cell activation']

pots_cond_wide = pots_conditions[pots_conditions['condition_name'].isin(top_conds)]
user_cond = pots_cond_wide.pivot_table(index='user_id', columns='condition_name', aggfunc='size', fill_value=0)
user_cond = (user_cond > 0).astype(int)

for c in top_conds:
    if c not in user_cond.columns:
        user_cond[c] = 0
user_cond = user_cond[top_conds]

cooc_matrix = user_cond.T.dot(user_cond)
cooc_pct = (cooc_matrix / 80 * 100).round(0)

label_map = {
    'pem': 'PEM', 'mcas': 'MCAS', 'dysautonomia': 'Dysautonomia',
    'me/cfs': 'ME/CFS', 'post-viral': 'Post-viral',
    'ehlers-danlos syndrome': 'EDS', 'ibs': 'IBS',
    'fibromyalgia': 'Fibromyalgia', 'small fiber neuropathy': 'SFN',
    'mast cell activation': 'Mast Cell Act.'
}
cooc_pct.index = [label_map.get(c, c) for c in cooc_pct.index]
cooc_pct.columns = [label_map.get(c, c) for c in cooc_pct.columns]

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(cooc_pct, dtype=bool), k=1)
sns.heatmap(cooc_pct, annot=True, fmt='.0f', cmap='YlOrRd',
            mask=mask, ax=ax, cbar_kws={'label': '% of POTS users (n=80)'},
            linewidths=0.5, vmin=0, vmax=80, annot_kws={'fontsize': 9})
ax.set_title('Condition Co-occurrence Among POTS Users\n(% of 80 POTS users with both conditions)',
             fontsize=12, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


**Key takeaway:** The diagonal shows prevalence of each condition among POTS users. The off-diagonal reveals clustering: the PEM-MCAS-dysautonomia-ME/CFS block is tightly interconnected (most cells above 40%). EDS co-occurs with POTS in about a third of cases. This supports the clinical concept of a post-infectious autonomic-immune phenotype.

## 4. Treatment Landscape: What POTS Users Try vs. the Broader Community

POTS users try more treatments, but do they try *different* treatments? And do those treatments work as well for them?

In [ ]:

CAUSAL_EXCLUSIONS = {'covid vaccine', 'flu vaccine', 'mmr vaccine', 'moderna vaccine',
                     'mrna covid-19 vaccine', 'pfizer vaccine', 'vaccine', 'vaccine injection',
                     'pfizer', 'booster'}

def get_group_treatments(group_ids, min_users=5):
    placeholders = ','.join(['?' for _ in group_ids])
    excl = GENERIC_TERMS | CAUSAL_EXCLUSIONS
    excl_ph = ','.join(['?' for _ in excl])
    df = pd.read_sql(f'''
        SELECT tr.user_id, t.canonical_name as drug, tr.sentiment,
               CASE tr.sentiment WHEN 'positive' THEN 1.0 WHEN 'mixed' THEN 0.5
                    WHEN 'neutral' THEN 0.0 WHEN 'negative' THEN -1.0 ELSE 0.0 END as score
        FROM treatment_reports tr
        JOIN treatment t ON tr.drug_id = t.id
        WHERE tr.user_id IN ({placeholders})
        AND LOWER(t.canonical_name) NOT IN ({excl_ph})
    ''', conn, params=list(group_ids) + list(excl))

    user_level = df.groupby(['drug', 'user_id']).agg(avg_score=('score', 'mean')).reset_index()
    user_level['outcome'] = user_level['avg_score'].apply(classify_outcome)

    summary = user_level.groupby('drug').agg(
        users=('user_id', 'nunique'),
        pos_rate=('outcome', lambda x: (x == 'positive').mean()),
        neg_rate=('outcome', lambda x: (x == 'negative').mean()),
        avg_score=('avg_score', 'mean')
    ).reset_index()
    summary = summary[summary['users'] >= min_users].sort_values('users', ascending=False)
    return summary

pots_tx = get_group_treatments(pots_ids, min_users=4)
nonpots_ids = set(pd.read_sql("SELECT DISTINCT user_id FROM treatment_reports", conn)['user_id']) - pots_ids
nonpots_tx = get_group_treatments(nonpots_ids, min_users=10)

comp = pots_tx.merge(nonpots_tx, on='drug', suffixes=('_pots', '_nonpots'), how='inner')
comp = comp.sort_values('users_pots', ascending=False).head(15)

for col_grp, n_col in [('pots', 'users_pots'), ('nonpots', 'users_nonpots')]:
    comp[f'{col_grp}_ci_lo'] = comp.apply(
        lambda r: wilson_ci(int(r[f'pos_rate_{col_grp}'] * r[n_col]), int(r[n_col]))[0], axis=1)
    comp[f'{col_grp}_ci_hi'] = comp.apply(
        lambda r: wilson_ci(int(r[f'pos_rate_{col_grp}'] * r[n_col]), int(r[n_col]))[1], axis=1)

display_comp = comp[['drug', 'users_pots', 'pos_rate_pots', 'users_nonpots', 'pos_rate_nonpots']].copy()
display_comp.columns = ['Treatment', 'POTS n', 'POTS +rate', 'non-POTS n', 'non-POTS +rate']
display_comp['POTS +rate'] = (display_comp['POTS +rate'] * 100).round(1)
display_comp['non-POTS +rate'] = (display_comp['non-POTS +rate'] * 100).round(1)

display(display_comp.style.set_caption('Treatment Positive Rates: POTS vs non-POTS (user-level, top shared treatments)')
    .hide(axis='index').format({'POTS +rate': '{:.1f}%', 'non-POTS +rate': '{:.1f}%'}))


**What this means:** Several patterns emerge: magnesium and electrolytes show near-universal positive sentiment in both groups. CoQ10 and nattokinase show lower positive rates in POTS users. However, all POTS-specific sample sizes are small (4-9 users), so individual treatment comparisons should be interpreted with extreme caution.

In [ ]:

plot_data = comp.sort_values('users_pots', ascending=True).tail(12)

fig, ax = plt.subplots(figsize=(12, 8))
y_pos = np.arange(len(plot_data))

ax.errorbar(plot_data['pos_rate_nonpots'] * 100, y_pos + 0.12,
            xerr=[(plot_data['pos_rate_nonpots'] - plot_data['nonpots_ci_lo']) * 100,
                  (plot_data['nonpots_ci_hi'] - plot_data['pos_rate_nonpots']) * 100],
            fmt='o', color='#95a5a6', markersize=7, capsize=3, label='non-POTS', linewidth=1.5)

ax.errorbar(plot_data['pos_rate_pots'] * 100, y_pos - 0.12,
            xerr=[(plot_data['pos_rate_pots'] - plot_data['pots_ci_lo']) * 100,
                  (plot_data['pots_ci_hi'] - plot_data['pos_rate_pots']) * 100],
            fmt='s', color='#3498db', markersize=7, capsize=3, label='POTS', linewidth=1.5)

ax.set_yticks(y_pos)
ax.set_yticklabels([f"{r['drug']}  (n={int(r['users_pots'])} vs {int(r['users_nonpots'])})"
                     for _, r in plot_data.iterrows()], fontsize=10)
ax.set_xlabel('Positive Rate (%)', fontsize=11)
ax.set_title('Treatment Positive Rates: POTS vs non-POTS\n(dots = point estimate, bars = 95% Wilson CI)',
             fontsize=12, fontweight='bold')
ax.axvline(50, color='red', linestyle='--', alpha=0.4, label='50% baseline')
ax.legend(loc='upper left', bbox_to_anchor=(0.0, -0.06), ncol=3, fontsize=10)
ax.set_xlim(-5, 105)
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()


**Key takeaway:** The wide confidence intervals for POTS users (blue squares) reflect small sample sizes. The chart identifies candidate treatments for further study -- treatments where the POTS point estimate diverges from non-POTS, even if we cannot yet confirm the difference statistically.

## 5. Overall Treatment Sentiment: Do POTS Users Report Worse Outcomes?

Individual treatment comparisons are underpowered. Pooling all treatment reports gives us a larger sample to test whether POTS users report systematically different outcomes.

In [ ]:

import math

excl_list = GENERIC_TERMS | CAUSAL_EXCLUSIONS
excl_quoted = ','.join([f"'{g}'" for g in excl_list])

all_reports = pd.read_sql(f'''
    SELECT tr.user_id, tr.sentiment, t.canonical_name,
           CASE tr.sentiment WHEN 'positive' THEN 1.0 WHEN 'mixed' THEN 0.5
                WHEN 'neutral' THEN 0.0 WHEN 'negative' THEN -1.0 ELSE 0.0 END as score
    FROM treatment_reports tr
    JOIN treatment t ON tr.drug_id = t.id
    WHERE LOWER(t.canonical_name) NOT IN ({excl_quoted})
''', conn)

all_reports['group'] = all_reports['user_id'].apply(lambda x: 'POTS' if x in pots_ids else 'non-POTS')

user_scores = all_reports.groupby(['user_id', 'group']).agg(
    avg_score=('score', 'mean'), n_reports=('sentiment', 'count')
).reset_index()
user_scores['outcome'] = user_scores['avg_score'].apply(classify_outcome)

for grp in ['POTS', 'non-POTS']:
    subset = user_scores[user_scores['group'] == grp]
    pos_n = (subset['outcome'] == 'positive').sum()
    total_n = len(subset)
    ci_lo, ci_hi = wilson_ci(pos_n, total_n)
    display(HTML(f"<b>{grp}</b>: {pos_n}/{total_n} positive ({pos_n/total_n*100:.1f}%), "
                 f"95% CI [{ci_lo*100:.1f}%, {ci_hi*100:.1f}%]"))

pots_scores = user_scores[user_scores['group'] == 'POTS']
nonpots_scores = user_scores[user_scores['group'] == 'non-POTS']

pots_pos = (pots_scores['outcome'] == 'positive').sum()
pots_neg_count = len(pots_scores) - pots_pos
nonpots_pos = (nonpots_scores['outcome'] == 'positive').sum()
nonpots_neg_count = len(nonpots_scores) - nonpots_pos

table_2x2 = [[pots_pos, pots_neg_count], [nonpots_pos, nonpots_neg_count]]
odds_ratio, p_fisher = fisher_exact(table_2x2)

p1 = pots_pos / (pots_pos + pots_neg_count)
p2 = nonpots_pos / (nonpots_pos + nonpots_neg_count)
cohens_h = 2 * (math.asin(math.sqrt(p1)) - math.asin(math.sqrt(p2)))

u_stat, p_mw = mannwhitneyu(pots_scores['avg_score'], nonpots_scores['avg_score'], alternative='two-sided')
n1, n2 = len(pots_scores), len(nonpots_scores)
rbc = 1 - (2 * u_stat) / (n1 * n2)

display(HTML(f"<br><b>Fisher exact test</b> (positive vs not): OR={odds_ratio:.2f}, p={p_fisher:.4f}"))
display(HTML(f"<b>Cohen's h</b>: {cohens_h:.3f} ({'small' if abs(cohens_h) < 0.3 else 'medium' if abs(cohens_h) < 0.8 else 'large'} effect)"))
display(HTML(f"<b>Mann-Whitney U</b> (user-level avg scores): U={u_stat:.0f}, p={p_mw:.4f}, rank-biserial r={rbc:.3f}"))


In [ ]:

sent_dist = all_reports.groupby(['group', 'sentiment']).size().unstack(fill_value=0)
sent_pct = sent_dist.div(sent_dist.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(10, 5))
sent_order = ['positive', 'mixed', 'neutral', 'negative']
colors_sent = {'positive': '#2ecc71', 'mixed': '#f39c12', 'neutral': '#95a5a6', 'negative': '#e74c3c'}
bottom = np.zeros(len(sent_pct))

for sent in sent_order:
    if sent in sent_pct.columns:
        vals = sent_pct[sent].values
        ax.barh(sent_pct.index, vals, left=bottom, color=colors_sent[sent],
                label=sent.capitalize(), edgecolor='white', height=0.5)
        for j, (v, b) in enumerate(zip(vals, bottom)):
            if v > 5:
                ax.text(b + v/2, j, f'{v:.1f}%', ha='center', va='center',
                        fontsize=10, fontweight='bold', color='white')
        bottom += vals

ax.set_xlabel('Percentage of Treatment Reports', fontsize=11)
ax.set_title('Sentiment Distribution: POTS vs non-POTS (report-level)', fontsize=12, fontweight='bold')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=10)
ax.set_xlim(0, 100)

for i, grp in enumerate(sent_pct.index):
    n_reports = sent_dist.loc[grp].sum()
    ax.text(101, i, f'n={n_reports}', va='center', fontsize=9, color='#555')

plt.tight_layout()
plt.show()


**What this means:** POTS users have a lower positive rate and higher negative rate than the broader community. The effect size is small to medium. This could reflect harder-to-treat symptoms, the fact that POTS users try a wider variety of treatments (including ones that do not work), or both.

## 6. Symptom Burden: What POTS Users Talk About

Text mining of post content reveals which symptoms and themes appear more frequently in POTS user posts compared to the broader community.

In [ ]:

themes = {
    'Fatigue': ['fatigu'],
    'Pain': ['pain'],
    'Brain fog': ['brain fog'],
    'Anxiety': ['anxiet', 'anxious'],
    'Tachycardia/HR': ['tachycardia', 'heart rate', 'hr spike', 'rapid heart'],
    'Dizziness': ['dizz', 'lightheaded', 'presyncop'],
    'Exercise intolerance': ['exercis', 'workout', 'cardio', 'deconditioning'],
    'Standing/orthostatic': ['stand', 'upright', 'orthostatic'],
    'Sleep issues': ['insomnia', 'sleep'],
    'GI symptoms': ['nausea', 'gastropar', 'bloat', 'diarr'],
    'Depression': ['depress'],
    'Doctor/medical': ['doctor', 'specialist', 'cardiolog', 'neurolog'],
    'Dismissed by provider': ['dismiss', 'not real', 'all in your head', 'gasligh'],
    'Recovery mentions': ['recover', 'improv', 'getting better', 'remission']
}

results_themes = []
for theme, keywords in themes.items():
    like_clauses = ' OR '.join([f"LOWER(p.body_text) LIKE '%{kw}%'" for kw in keywords])

    pots_n = conn.execute(f'''
        SELECT COUNT(DISTINCT p.user_id) FROM posts p
        WHERE p.user_id IN (SELECT DISTINCT user_id FROM conditions WHERE LOWER(condition_name) LIKE '%pots%')
        AND ({like_clauses})
    ''').fetchone()[0]

    nonpots_n = conn.execute(f'''
        SELECT COUNT(DISTINCT p.user_id) FROM posts p
        WHERE p.user_id NOT IN (SELECT DISTINCT user_id FROM conditions WHERE LOWER(condition_name) LIKE '%pots%')
        AND ({like_clauses})
    ''').fetchone()[0]

    pots_pct = pots_n / 80 * 100
    nonpots_pct = nonpots_n / 2747 * 100

    table_t = [[pots_n, 80 - pots_n], [nonpots_n, 2747 - nonpots_n]]
    or_t, p_t = fisher_exact(table_t)

    results_themes.append({
        'Theme': theme,
        'POTS (n=80)': f"{pots_n} ({pots_pct:.1f}%)",
        'non-POTS (n=2747)': f"{nonpots_n} ({nonpots_pct:.1f}%)",
        'Fold enrichment': round(pots_pct / max(nonpots_pct, 0.1), 1),
        'OR': round(or_t, 2),
        'p_raw': p_t
    })

theme_df = pd.DataFrame(results_themes).sort_values('Fold enrichment', ascending=False)
theme_df['p-value'] = theme_df['p_raw'].apply(lambda x: f"{x:.2e}" if x < 0.001 else f"{x:.4f}")
theme_df['Significant'] = theme_df['p_raw'].apply(lambda x: 'Yes' if x < 0.05 else 'No')

display(theme_df[['Theme', 'POTS (n=80)', 'non-POTS (n=2747)', 'Fold enrichment', 'OR', 'p-value', 'Significant']].style
    .set_caption('Text Theme Prevalence: POTS vs non-POTS Users').hide(axis='index'))


In [ ]:

plot_themes = pd.DataFrame(results_themes).sort_values('Fold enrichment', ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
y_pos = np.arange(len(plot_themes))
colors_dot = ['#e74c3c' if p < 0.05 else '#95a5a6' for p in plot_themes['p_raw']]

ax.scatter(plot_themes['Fold enrichment'], y_pos, c=colors_dot, s=100, zorder=3,
           edgecolors='white', linewidth=0.5)
ax.barh(y_pos, plot_themes['Fold enrichment'],
        color=[c + '30' for c in colors_dot], height=0.4, zorder=1)

ax.set_yticks(y_pos)
ax.set_yticklabels(plot_themes['Theme'], fontsize=10)
ax.set_xlabel('Fold Enrichment (POTS vs non-POTS)', fontsize=11)
ax.set_title('Symptom and Experience Theme Enrichment in POTS Users\n(red = p<0.05, grey = not significant)',
             fontsize=12, fontweight='bold')
ax.axvline(1.0, color='black', linestyle='--', alpha=0.3)

for i, (_, row) in enumerate(plot_themes.iterrows()):
    ax.text(row['Fold enrichment'] + 0.08, i, f"{row['Fold enrichment']:.1f}x",
            va='center', fontsize=9, color='#333')

plt.tight_layout()
plt.show()


**Key takeaway:** Nearly every symptom theme is enriched in POTS users. Dizziness/lightheadedness shows the highest enrichment (~5x), consistent with the core POTS phenotype. Standing/orthostatic symptoms and tachycardia are also elevated. Non-autonomic symptoms like pain, fatigue, and brain fog are enriched at 2-3x, suggesting POTS users experience broader symptom burden beyond their core autonomic dysfunction.

## 7. POTS-Targeted Treatments: Standard-of-Care Performance

Clinical guidelines for POTS recommend beta blockers, ivabradine, midodrine, along with lifestyle measures (electrolytes, salt loading, compression). How do these perform in community reports?

In [ ]:

pots_uid_list = ','.join([f"'{uid}'" for uid in pots_ids])
pots_specific = pd.read_sql(f'''
    SELECT tr.user_id, t.canonical_name as drug, tr.sentiment,
           CASE tr.sentiment WHEN 'positive' THEN 1.0 WHEN 'mixed' THEN 0.5
                WHEN 'neutral' THEN 0.0 WHEN 'negative' THEN -1.0 ELSE 0.0 END as score
    FROM treatment_reports tr
    JOIN treatment t ON tr.drug_id = t.id
    WHERE tr.user_id IN ({pots_uid_list})
''', conn)

def categorize_pots_tx(name):
    nl = name.lower()
    if any(x in nl for x in ['propranolol', 'metoprolol', 'beta blocker', 'atenolol']):
        return 'Beta blockers'
    elif 'ivabradine' in nl or 'corlanor' in nl:
        return 'Ivabradine'
    elif 'midodrine' in nl:
        return 'Midodrine'
    elif any(x in nl for x in ['electrolyte', 'salt', 'sodium']):
        return 'Electrolytes/salt'
    elif 'magnesium' in nl:
        return 'Magnesium'
    elif any(x in nl for x in ['ketotifen', 'famotidine', 'cetirizine', 'fexofenadine',
                                 'h1 antihistamine', 'h2 antihistamine']):
        return 'Antihistamines (MCAS)'
    elif 'low dose naltrexone' in nl or nl == 'ldn':
        return 'Low Dose Naltrexone'
    elif any(x in nl for x in ['coq10', 'ubiquinol']):
        return 'CoQ10'
    elif 'nattokinase' in nl:
        return 'Nattokinase'
    elif 'probiotics' in nl:
        return 'Probiotics'
    else:
        return None

pots_specific['category'] = pots_specific['drug'].apply(categorize_pots_tx)
pots_categorized = pots_specific[pots_specific['category'].notna()].copy()

user_cat = pots_categorized.groupby(['category', 'user_id']).agg(
    avg_score=('score', 'mean')).reset_index()
user_cat['outcome'] = user_cat['avg_score'].apply(classify_outcome)

cat_summary = user_cat.groupby('category').agg(
    users=('user_id', 'nunique'),
    pos_n=('outcome', lambda x: (x == 'positive').sum()),
    neg_n=('outcome', lambda x: (x == 'negative').sum()),
).reset_index()
cat_summary['pos_rate'] = cat_summary['pos_n'] / cat_summary['users']
cat_summary['ci_lo'] = cat_summary.apply(lambda r: wilson_ci(int(r['pos_n']), int(r['users']))[0], axis=1)
cat_summary['ci_hi'] = cat_summary.apply(lambda r: wilson_ci(int(r['pos_n']), int(r['users']))[1], axis=1)
cat_summary = cat_summary.sort_values('users', ascending=False)

cat_summary['binom_p'] = cat_summary.apply(
    lambda r: binomtest(int(r['pos_n']), int(r['users']), 0.5).pvalue, axis=1)

disp = cat_summary[['category', 'users', 'pos_n', 'neg_n', 'pos_rate', 'ci_lo', 'ci_hi', 'binom_p']].copy()
disp.columns = ['Treatment Class', 'Users', 'Positive', 'Negative', 'Pos Rate', 'CI Low', 'CI High', 'p (vs 50%)']
disp['Pos Rate'] = (disp['Pos Rate'] * 100).round(1)
disp['CI Low'] = (disp['CI Low'] * 100).round(1)
disp['CI High'] = (disp['CI High'] * 100).round(1)
disp['p (vs 50%)'] = disp['p (vs 50%)'].apply(lambda x: f"{x:.4f}" if x >= 0.001 else f"{x:.2e}")

display(disp.style.set_caption('POTS-Relevant Treatment Classes (user-level, POTS users only)').hide(axis='index'))


In [ ]:

plot_cat = cat_summary.sort_values('pos_rate', ascending=True).copy()

fig, ax = plt.subplots(figsize=(11, 6))
y_pos = np.arange(len(plot_cat))

bars = ax.barh(y_pos, plot_cat['pos_rate'] * 100, height=0.5,
               color='#2ecc71', label='Positive rate', edgecolor='white')
ax.errorbar(plot_cat['pos_rate'] * 100, y_pos,
            xerr=[(plot_cat['pos_rate'] - plot_cat['ci_lo']) * 100,
                  (plot_cat['ci_hi'] - plot_cat['pos_rate']) * 100],
            fmt='none', color='black', capsize=3, linewidth=1)

ax.set_yticks(y_pos)
ax.set_yticklabels([f"{r['category']}  (n={int(r['users'])})" for _, r in plot_cat.iterrows()], fontsize=10)
ax.set_xlabel('Positive Rate (%) with 95% Wilson CI', fontsize=11)
ax.set_title('POTS-Relevant Treatment Classes: Positive Rate Among POTS Users',
             fontsize=12, fontweight='bold')
ax.axvline(50, color='red', linestyle='--', alpha=0.4, label='50% baseline')
ax.set_xlim(0, 105)
ax.legend(loc='upper left', bbox_to_anchor=(0.0, -0.08), ncol=2, fontsize=10)

plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()


**What this means:** Among POTS users, magnesium, electrolytes/salt, and probiotics show the highest positive rates (above 80%). Antihistamines and LDN also perform well. Beta blockers and nattokinase show more mixed results. The wide confidence intervals throughout emphasize these are preliminary signals. Electrolytes and magnesium performing well aligns with clinical expectations for volume depletion in POTS. The strong antihistamine showing is consistent with the 75% MCAS comorbidity rate.

## 8. Counterintuitive Findings Worth Investigating

In [ ]:

# Sensitivity check: does POTS-worse-outcomes hold when dropping extremes?
pots_user_scores = user_scores[user_scores['group'] == 'POTS'].copy()
pots_sorted = pots_user_scores.sort_values('avg_score')
trimmed_pots = pots_sorted.iloc[3:-3]
trimmed_nonpots = user_scores[user_scores['group'] == 'non-POTS']
u_trim, p_trim = mannwhitneyu(trimmed_pots['avg_score'], trimmed_nonpots['avg_score'], alternative='two-sided')

display(HTML(f"<b>Sensitivity check (dropping 3 most extreme POTS users each direction):</b> "
             f"Mann-Whitney p={p_trim:.4f} -- "
             f"{'main finding holds' if p_trim < 0.05 else 'finding becomes non-significant, suggesting fragility'}"))

# Beta blocker analysis
bb_pots = pots_specific[pots_specific['drug'].str.lower().str.contains('propranolol|metoprolol|beta blocker|atenolol')].copy()
bb_user = bb_pots.groupby('user_id').agg(avg_score=('score', 'mean')).reset_index()
bb_user['outcome'] = bb_user['avg_score'].apply(classify_outcome)
bb_pos = (bb_user['outcome'] == 'positive').sum()
bb_total = len(bb_user)
if bb_total > 0:
    bb_ci = wilson_ci(bb_pos, bb_total)
    display(HTML(f"<br><b>Beta blocker sentiment among POTS users:</b> {bb_pos}/{bb_total} positive "
                 f"({bb_pos/bb_total*100:.0f}%), 95% CI [{bb_ci[0]*100:.1f}%, {bb_ci[1]*100:.1f}%]"))

elec_pots = pots_specific[pots_specific['drug'].str.lower().str.contains('electrolyte|salt|sodium')].copy()
elec_user = elec_pots.groupby('user_id').agg(avg_score=('score', 'mean')).reset_index()
elec_user['outcome'] = elec_user['avg_score'].apply(classify_outcome)
elec_pos = (elec_user['outcome'] == 'positive').sum()
elec_total = len(elec_user)
if elec_total > 0:
    display(HTML(f"<b>Electrolyte/salt sentiment among POTS users:</b> {elec_pos}/{elec_total} positive "
                 f"({elec_pos/elec_total*100:.0f}%)"))


**Finding 1: Beta blockers underperform their clinical reputation in community sentiment.** Beta blockers are first-line therapy for POTS, yet their community positive rate among POTS users is modest -- below electrolytes, magnesium, and antihistamines. This does not mean beta blockers are ineffective. It likely reflects that (a) community reports capture side effects and dose titration alongside efficacy, (b) beta blockers are prescribed to the sickest patients who may have harder-to-treat symptoms, and (c) lifestyle interventions like electrolytes have a more straightforward positive experience. The gap between clinical guideline confidence and community sentiment deserves follow-up with larger samples.

**Finding 2: POTS users carry dramatically more comorbidities (median 8 vs 2) but their treatment positive rate is only modestly lower (~10 percentage points).** Given the much higher disease complexity, one might expect substantially worse outcomes. The relatively small gap could suggest that POTS users are finding effective combinations, that their conditions are highly overlapping (treating one helps another), or that their higher engagement and health literacy help them find effective treatments faster.

No other patterns met the bar for genuinely counterintuitive findings. The comorbidity clustering and symptom enrichment patterns are well-known and expected.

## 9. What Patients Are Saying *(experimental -- under development)*

In [ ]:

quotes_raw = pd.read_sql('''
    SELECT SUBSTR(p.body_text, 1, 500) as text, date(p.post_date, 'unixepoch') as dt
    FROM posts p
    WHERE p.user_id IN (
        SELECT DISTINCT user_id FROM conditions WHERE LOWER(condition_name) LIKE '%pots%'
    )
    AND LENGTH(p.body_text) > 80
    AND (LOWER(p.body_text) LIKE '%ivabradine%'
         OR LOWER(p.body_text) LIKE '%beta blocker%'
         OR LOWER(p.body_text) LIKE '%propranolol%'
         OR LOWER(p.body_text) LIKE '%electrolyte%'
         OR LOWER(p.body_text) LIKE '%magnesium%'
         OR LOWER(p.body_text) LIKE '%antihistamine%'
         OR LOWER(p.body_text) LIKE '%ketotifen%'
         OR LOWER(p.body_text) LIKE '%naltrexone%')
    ORDER BY p.post_date DESC
    LIMIT 30
''', conn)

html_q = '<style>.quote-box { background: #f8f9fa; border-left: 3px solid #3498db; padding: 10px 15px; margin: 10px 0; font-style: italic; }</style>'

displayed = 0
cats_shown = set()
for _, row in quotes_raw.iterrows():
    if displayed >= 5:
        break
    text = row['text'].strip()
    if len(text) < 80:
        continue
    tl = text.lower()

    cat = None
    if 'ivabradine' in tl and 'Ivabradine' not in cats_shown:
        cat = 'Ivabradine experience'
        cats_shown.add('Ivabradine')
    elif ('beta blocker' in tl or 'propranolol' in tl) and 'Beta' not in cats_shown:
        cat = 'Beta blocker experience'
        cats_shown.add('Beta')
    elif 'electrolyte' in tl and 'Electrolyte' not in cats_shown:
        cat = 'Electrolyte/salt loading'
        cats_shown.add('Electrolyte')
    elif ('antihistamine' in tl or 'ketotifen' in tl) and 'AH' not in cats_shown:
        cat = 'Antihistamine (MCAS management)'
        cats_shown.add('AH')
    elif ('magnesium' in tl or 'naltrexone' in tl) and 'Supp' not in cats_shown:
        cat = 'Supplement/LDN experience'
        cats_shown.add('Supp')

    if cat is None:
        continue

    words = text.split()
    snippet = ' '.join(words[:40])
    if len(words) > 40:
        snippet += '...'

    html_q += f'<p><b>{cat}:</b></p><div class="quote-box">"{snippet}" <br><small>-- POTS user, {row["dt"]}</small></div>\n'
    displayed += 1

display(HTML(html_q))


These quotes illustrate the diversity of POTS treatment experiences -- from dramatic improvement with specific interventions to long trial-and-error processes across multiple treatment classes.

## 10. Tiered Recommendations

Recommendations are tiered by evidence strength from this dataset. All are based on community sentiment from POTS-identified users, not clinical trial data.

In [ ]:

recs = []
for _, row in cat_summary.iterrows():
    n = int(row['users'])
    pos = int(row['pos_n'])
    rate = row['pos_rate']
    ci = (row['ci_lo'], row['ci_hi'])
    p_b = row['binom_p']

    if n >= 30 and p_b < 0.05:
        tier = 'Strong'
    elif n >= 15 and p_b < 0.10:
        tier = 'Moderate'
    elif n >= 5:
        tier = 'Preliminary'
    else:
        tier = 'Insufficient'

    direction = 'Positive' if rate > 0.5 else 'Mixed/negative' if rate < 0.5 else 'Neutral'
    recs.append({'Treatment': row['category'], 'n': n, 'Pos Rate': f"{rate*100:.0f}%",
                 'CI': f"[{ci[0]*100:.0f}%, {ci[1]*100:.0f}%]", 'Tier': tier, 'Signal': direction})

recs_df = pd.DataFrame(recs).sort_values(['Tier', 'n'], ascending=[True, False])

for tier in ['Strong', 'Moderate', 'Preliminary', 'Insufficient']:
    subset = recs_df[recs_df['Tier'] == tier]
    if len(subset) > 0:
        display(HTML(f"<h4>{tier} Evidence</h4>"))
        display(subset[['Treatment', 'n', 'Pos Rate', 'CI', 'Signal']].style.hide(axis='index'))


In [ ]:

tier_colors = {'Strong': '#27ae60', 'Moderate': '#f39c12', 'Preliminary': '#95a5a6'}
tiers_to_show = [t for t in ['Strong', 'Moderate', 'Preliminary'] if len(recs_df[recs_df['Tier']==t]) > 0]

if len(tiers_to_show) > 0:
    fig, axes = plt.subplots(1, len(tiers_to_show), figsize=(14, 4),
                              gridspec_kw={'width_ratios': [max(1, len(recs_df[recs_df['Tier']==t])) for t in tiers_to_show]})
    if len(tiers_to_show) == 1:
        axes = [axes]

    for idx, tier in enumerate(tiers_to_show):
        ax = axes[idx]
        subset = recs_df[recs_df['Tier'] == tier].sort_values('n', ascending=True)
        if len(subset) == 0:
            ax.set_visible(False)
            continue

        yp = np.arange(len(subset))
        rates = subset['Pos Rate'].str.rstrip('%').astype(float)

        ax.barh(yp, rates, color=tier_colors[tier], height=0.6, edgecolor='white')
        ax.set_yticks(yp)
        ax.set_yticklabels([f"{r['Treatment']} (n={r['n']})" for _, r in subset.iterrows()], fontsize=9)
        ax.set_xlim(0, 105)
        ax.axvline(50, color='red', linestyle='--', alpha=0.3)
        ax.set_title(f'{tier}', fontsize=11, fontweight='bold', color=tier_colors[tier])
        ax.set_xlabel('Pos Rate %', fontsize=9)

        for j, (val, _) in enumerate(zip(rates, yp)):
            if val > 15:
                ax.text(val - 2, j, f'{val:.0f}%', ha='right', va='center', fontsize=9, color='white', fontweight='bold')

    fig.suptitle('Treatment Recommendations by Evidence Tier (POTS Users)', fontsize=12, fontweight='bold', y=1.05)
    plt.tight_layout()
    plt.show()


## 11. Conclusion

POTS patients in the Long COVID community are a distinct and identifiable subgroup. They are not representative of the broader community -- they are sicker, more engaged, and more treatment-experienced. With a median of 8 co-occurring conditions (vs 2 for non-POTS users), they represent the complex, multi-system end of the Long COVID spectrum.

The comorbidity data strongly supports the clinical concept of the POTS-MCAS-dysautonomia cluster. 75% of POTS users also report MCAS, and 65% report dysautonomia -- far higher than background rates in this community. This is not a new finding, but quantifying it in a patient community adds context: these patients are self-identifying these connections, which aligns with emerging clinical understanding of post-infectious autonomic dysfunction.

Treatment-wise, POTS users favor a combination approach: lifestyle interventions (electrolytes, magnesium), MCAS-targeted therapies (antihistamines like ketotifen and famotidine), and immune-modulating agents (LDN). These show the strongest positive sentiment. The clinical first-line treatment -- beta blockers -- shows only modest community sentiment, a pattern worth investigating with larger samples. This could reflect side-effect burden, indication bias, or that community reports capture the full treatment experience rather than efficacy alone.

The most actionable finding for patients: if you have POTS with Long COVID, screening for MCAS and dysautonomia is strongly supported by this data. The treatment approaches that work for these overlapping conditions (antihistamines, electrolytes) appear to have good community support. For researchers: the POTS-MCAS-dysautonomia triad deserves formal characterization as a Long COVID phenotype, and the beta blocker community sentiment gap deserves investigation with controlled data.

## 12. Research Limitations

1. **Selection bias:** Users of r/covidlonghaulers are self-selected and likely skew toward more severe, longer-duration illness. POTS patients who recovered quickly are underrepresented.

2. **Reporting bias:** Users are more likely to post about treatments that provoked strong reactions (positive or negative). Treatments that were unremarkable are underreported, inflating both tails.

3. **Survivorship bias:** We only observe users who are still active in the community. Those who recovered and left, or those too ill to post, are missing.

4. **Recall bias:** Treatment reports may reflect current perceptions rather than actual experiences, especially for treatments tried months ago.

5. **Confounding:** POTS users have more comorbidities and try more treatments. Attributing outcomes to any single treatment is unreliable -- improvements may reflect other concurrent treatments, natural disease course, or regression to the mean.

6. **No control group:** There is no untreated comparison group. The 50% baseline is arbitrary; the true spontaneous improvement rate for Long COVID symptoms is unknown and likely varies by symptom.

7. **Sentiment vs efficacy:** Community sentiment reflects the full treatment experience (efficacy, side effects, cost, access) rather than clinical efficacy alone. A treatment can be clinically effective but receive negative sentiment due to side effects.

8. **Temporal snapshot:** This data covers one month (March-April 2026). Treatment patterns, community composition, and sentiment may shift over time.

**Additional limitations specific to this analysis:**
- POTS identification is based on text mining, not verified diagnosis.
- Sample sizes for individual POTS treatments are very small (typically 3-9 users). All individual treatment findings are preliminary.
- This is a survey/preliminary study. No causal claims can be made.

In [ ]:

display(HTML('<div style="font-size: 1.2em; font-weight: bold; font-style: italic; margin-top: 30px; '
             'padding: 15px; border: 1px solid #ccc; background: #f9f9f9;">'
             'These findings reflect reporting patterns in online communities, not population-level '
             'treatment effects. This is not medical advice.</div>'))
